In [ ]:
import pandas as pd

In [ ]:
# Convert all HMMscan files to dataframes
# Pre-process .tblout file into a datafram with the following columns:

columns = [
    "target_name",
    "protein_name",
    "query_name",
    "full_sequence_evalue",
    "full_sequence_score",
    "full_sequence_bias",
    "best_domain_evalue",
    "best_domain_score",
    "best_domain_bias",
    "exp",
    "reg",
    "clu",
    "ov",
    "env",
    "dom",
    "rep",
    "inc"
]

In [1]:
# target name = HMM profile
# protein_name = Protein
# query_name = ID_Genus_species

# the remaining columns should be from the HMM .tblout file

In [ ]:
# list HMM training datasets here
target_names = []

def process_tblout_data(df, target_names):
    filtered_df = df[df['target_name'].isin(target_names)]

    species_counts = []

    for target in target_names:
        target_df = filtered_df[filtered_df['target_name'] == target]
        species_count = target_df['query_name'].nunique()
        species_counts.append(species_count)

    new_df = pd.DataFrame({
        'target_name': target_names,
        'species': species_counts,
    })

    new_df.sort_values(by='species', ascending=False, inplace=True)

    return new_df

In [ ]:
def most_significant_profile(df):
    """
    Filter DataFrame to keep only the rows with the lowest full_sequence_evalue for each query_name.

    Parameters:
    df (pandas.DataFrame): DataFrame containing data from the tblout file.

    Returns:
    pandas.DataFrame: DataFrame with only the most significant profile for each query_name.
    """
    # Convert 'full_sequence_evalue' column to numeric, ignoring errors
    df['full_sequence_evalue'] = pd.to_numeric(df['full_sequence_evalue'], errors='coerce')

    # Drop rows with missing values in 'full_sequence_evalue' column
    df = df.dropna(subset=['full_sequence_evalue'])

    # Get indices of rows with the lowest 'full_sequence_evalue' for each 'query_name'
    min_indices = df.groupby('query_name')['full_sequence_evalue'].idxmin()

    # Filter the DataFrame to keep only the rows with the lowest 'full_sequence_evalue' for each 'query_name'
    df_filtered = df.loc[min_indices]

    return df_filtered


In [ ]:
def significant_profile_count(df):
    """
    Count occurrences of each unique target_name in the DataFrame.

    Parameters:
    df (pandas.DataFrame): DataFrame containing data from the tblout file.

    Returns:
    None
    """
    
    # Get unique entries from the 'target_name' column
    unique_entries = df['target_name'].unique()
    # Sort the unique entries
    unique_entries.sort()
    
    # Initialize an empty DataFrame to store the results
    result_df = pd.DataFrame(columns=['profile', 'count', 'percentage'])


    # Iterate through each unique entry
    total_count = df.shape[0]
    for entry in unique_entries:
        # Count occurrences of the current entry
        count = df[df['target_name'] == entry].shape[0]
        # Calculate the percentage of count relative to the total count
        percentage = (count / total_count) * 100
        # Round the percentage to 2 decimal places
        percentage = round(percentage, 2)
        # Append the entry, count, and percentage to the result DataFrame
        result_df = result_df.append({'profile': entry, 'count': count, 'percentage': percentage}, ignore_index=True)

        
    # Sort the DataFrame by the 'species' column
    result_df.sort_values(by='count', ascending=False, inplace=True)
    

    return result_df
